# 04 - Compare HotCRP and Researchr Sources

In this notebook, I compare HotCRP lists with the visible conference website
lists.

For each conference year, I check whether HotCRP is closer to:

- Researchr PC only
- Researchr PC + EPC
- Researchr PC + ERC
- Researchr PC + EPC + ERC

This helps me understand what HotCRP is measuring.


## 1 - Setup

In [11]:
import re
import unicodedata
from difflib import SequenceMatcher
from pathlib import Path

import pandas as pd


In [12]:
import sys
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "project_setup.py").exists() and (candidate / "config" / "project_config.yaml").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break
else:
    raise RuntimeError(
        "Could not find the repository root. Launch Jupyter from the repo root "
        "or set PYTHONPATH to the folder containing project_setup.py."
    )

from project_setup import setup_project

setup = setup_project()
project_folder = setup.project_folder
PROJECT = project_folder
repo = project_folder
config_path = setup.config_path
project_config = setup.project_config

run_mode = setup.run_mode
inputs_config = setup.inputs
outputs_config = setup.outputs
openalex_config = setup.openalex

allow_network = setup.allow_network
use_existing_data = setup.use_existing_data
overwrite_data = setup.overwrite_data
overwrite_artifacts = setup.overwrite_artifacts
openalex_sample_limit = setup.openalex_sample_limit
openalex_sample_include_work_ids = setup.openalex_sample_include_work_ids

step_1_data_dir = project_folder / "step_1_data"
step_1_artifacts_dir = project_folder / "step_1_artifacts"
intermediate_dir = step_1_data_dir / "intermediate"
prepared_dir = step_1_data_dir / "prepared"
summary_tables_dir = step_1_artifacts_dir / "summary_tables"
dependency_tables_dir = step_1_artifacts_dir / "dependency_tables"
check_tables_dir = step_1_artifacts_dir / "check_tables"

intermediate_dir.mkdir(parents=True, exist_ok=True)
prepared_dir.mkdir(parents=True, exist_ok=True)
summary_tables_dir.mkdir(parents=True, exist_ok=True)
dependency_tables_dir.mkdir(parents=True, exist_ok=True)
check_tables_dir.mkdir(parents=True, exist_ok=True)

print(project_folder)
print(f"Run mode: {run_mode}")


/Users/endersari/2026-02-citations-vs-pc-memberships
Run mode: fast


## 2 - Name Map

In [13]:
fold_map = str.maketrans({
    "ø": "o", "Ø": "O",
    "æ": "ae", "Æ": "AE",
    "œ": "oe", "Œ": "OE",
    "ß": "ss",
    "ł": "l", "Ł": "L",
    "đ": "d", "Đ": "D",
    "þ": "th", "Þ": "Th",
    "ð": "d", "Ð": "D",
    "’": "'", "‘": "'",
    "“": '"', "”": '"',
})


def normalize_name(name):
    if not isinstance(name, str) or not name:
        return ""
    name = unicodedata.normalize("NFKD", name)
    name = "".join(ch for ch in name if not unicodedata.combining(ch))
    name = name.translate(fold_map)
    name = name.lower()
    name = re.sub(r"\([^)]*\)", " ", name)
    name = re.sub(r"[^a-z0-9 ]+", " ", name)
    name = re.sub(r"\s+", " ", name).strip()
    return name


NAME_MAP = {
    "Nguyen Kim": "Kim Nguyễn",
    "Simon Peyton-Jones": "Simon Peyton Jones",
    "Madhusudan Parthasarathy": "P. Madhusudan",
    "Max Schäfer": "Max Schaefer",
    "Andy Gordon": "Andrew D. Gordon",
    "Armando The POPL Chair": "Armando Solar-Lezama",
    "Casper Bach Poulsen": "Casper Bach",
    "Jenna DiVincenzo": "Jenna DiVincenzo (Wise)",
    "Mangpo Phothilimthana": "Phitchaya Mangpo Phothilimthana",
    "Alastair Donaldson": "Alastair F. Donaldson",
    "Angelica Moreira": "Angélica Aparecida Moreira",
    "Amir Kafshdar Goharshady": "Amir K. Goharshady",
    "Andrew Pitts": "Andrew M. Pitts",
    "Bruno Oliveira": "Bruno C. d. S. Oliveira",
    "Colin S. Gordon": "Colin Gordon",
    "Corina Pasareanu": "Corina S. Păsăreanu",
    "Dan Licata": "Daniel R. Licata",
    "Daniel W. Barowy": "Dan Barowy",
    "David Pearce": "David J. Pearce",
    "David Sands": "Dave Sands",
    "Earl Barr": "Earl T. Barr",
    "Feras Saad": "Feras A. Saad",
    "Fernando Pereira": "Fernando Magno Quintão Pereira",
    "G Ramalingam": "G. Ramalingam",
    "Gary Tan": "Gang (Gary) Tan",
    "Hoan Nguyen": "Hoan Anh Nguyen",
    "Isil Dillig": "Işıl Dillig",
    "Jan Rellermeyer": "Jan S. Rellermeyer",
    "Jeffrey Foster": "Jeffrey S. Foster",
    "Jennifer Sartor": "Jennifer B. Sartor",
    "Jeremy Siek": "Jeremy G. Siek",
    "Joanna Cecilia da Silva Santos": "Joanna C. S. Santos",
    "Jon Sterling": "Jonathan Sterling",
    "Kathryn Gray": "Kathryn E. Gray",
    "Kathryn S. McKinley": "Kathryn S McKinley",
    "Ken McMillan": "Kenneth L. McMillan",
    "Magnus Myreen": "Magnus O. Myreen",
    "Martin T. Vechev": "Martin Vechev",
    "Max New": "Max S. New",
    "Philiip Wadler": "Philip Wadler",
    "Richard Eisenberg": "Richard A. Eisenberg",
    "Robby Findler": "Robert Bruce Findler",
    "Stefan K Muller": "Stefan K. Muller",
    "Stephen Fink": "Stephen J Fink",
    "Tien Nguyen": "Tien N. Nguyen",
    "Uday Khedker": "Uday P. Khedker",
    "Umut Acar": "Umut A. Acar",
    "Bob Atkey": "Robert Atkey",
    "Garrett Morris": "J. Garrett Morris",
    "Harley Eades": "Harley D. Eades III",
    "Harry Xu": "Guoqing Harry Xu",
    "Aditya Thakur": "Aditya V. Thakur",
    "Andrzej S. Murawski": "Andrzej Murawski",
    "Anthony W. Lin": "Anthony Widjaja Lin",
    "Ben Titzer": "Ben L. Titzer",
    "Ben Livshits": "Benjamin Livshits",
    "Benjamin Kaminski": "Benjamin Lucien Kaminski",
    "Charlie Murphy": "Charlie Murphy",
    "Cristina Lopes": "Crista Lopes",
    "Dalal Alrajeh": "Dalal Alrajeh",
    "David Bacon": "David F. Bacon",
    "David Christiansen": "David Thrane Christiansen",
    "Daniela Petrisan": "Daniela Petrişan",
    "Dimitri Racordon": "Dimi Racordon",
    "Emery Berger": "Emery D. Berger",
    "Erika Abraham": "Erika Ábrahám",
    "Guilhem Jaber": "Guilhème Jaber",
    "Guy Gueta": "Guy Golan-Gueta",
    "Guy Steele": "Guy L. Steele Jr.",
    "Guy L. Steele": "Guy L. Steele Jr.",
    "James Wilcox": "James R. Wilcox",
    "Joshua Dunfield": "Jana Dunfield",
    "Konstantinos Sagonas": "Konstantinos (Kostis) Sagonas",
    "Laura Castro": "Laura M. Castro",
    "Matthew Parkinson": "Matthew J. Parkinson",
    "Michael Bond": "Michael D. Bond",
    "Michael O'Boyle": "Michael F. P. O'Boyle",
    "Mukund Raghotaman": "Mukund Raghothaman",
    "Nick Smallbone": "Nicholas Smallbone",
    "Ras Bodik": "Rastislav Bodík",
    "Ryan Newton": "Ryan R. Newton",
    "Sarah Chasins": "Sarah E. Chasins",
    "Sharon Shoham": "Sharon Shoham Buchbinder",
    "Shuvendu Lahiri": "Shuvendu K. Lahiri",
    "Stefan Muller": "Stefan K. Muller",
    "Stephen Freund": "Stephen N. Freund",
    "Thomas Jensen": "Thomas P. Jensen",
    "Thomas Würthinger": "Thomas Wuerthinger",
    "Trevor McDonell": "Trevor L. McDonell",
    "V. Krishna Nandivada": "V Krishna Nandivada",
    "Vikash Mansinghka": "Vikash K. Mansinghka",
    "Will Byrd": "William E. Byrd",
    "Daniel Licata": "Daniel R. Licata",
    "Neelakantan Krishnaswami": "Neel Krishnaswami",
    "Cristina V. Lopes": "Crista Lopes",
    "Madanlal Musuvathi": "Madan Musuvathi",
    "Martin Rinard": "Martin C. Rinard",
    "Michelle Strout": "Michelle Mills Strout",
    "Sam Guyer": "Samuel Z. Guyer",
    "Sara Baghsorkni": "Sara Baghsorkhi",
    "Thomas Wenisch": "Thomas F. Wenisch",
    "Tom Ball": "Thomas Ball",
    "Vikram Adve": "Vikram S. Adve",
    "Zach Tatlock": "Zachary Tatlock",
    "Nathan Foster": "Nate Foster",
    "Hans Boehm": "Hans-J. Boehm",
    "Jeff Foster": "Jeffrey S. Foster",
    "Julien Verlaguet": "Julien Verlaguet",
    "William Byrd": "William E. Byrd",
    "Alexander Lew": "Alexander K. Lew",
    "Bill Harris": "William Harris",
    "Gilbert Bernstein": "Gilbert Louis Bernstein",
    "Jorge Navas": "Jorge A. Navas",
    "Mahmut Kandemir": "Mahmut Taylan Kandemir",
    "Kristóf Marussy": "Kristóf Marussy",
    "Laura Kovacs": "Laura Kovács",
    "Liang-Ting Chen": "Liang-Ting Chen",
    "Marie Kerjean": "Marie Kerjean",
    "Muralidaran Vijayaraghavan": "Murali Vijayaraghavan",
    "Pavel Parizek": "Pavel Parízek",
    "Peng Fu": "Frank Fu",
    "Rishiyur Nikhil": "Rishiyur S. Nikhil",
    "Sonia Marin": "Sonia Marı́n",
    "Suparna Bhattacharya": "Suparna Bhattacharya",
    "Tomas Petricek": "Tomas Petricek",
    "Vaishnavi Sundararajan": "Vaishnavi Sundararajan",
    "Yatin Manerkar": "Yatin A. Manerkar",
    "Zheng Zhang": "Eddy (Zheng) Zhang",
}

name_map_norm = {normalize_name(k): v for k, v in NAME_MAP.items()}


def lookup_name(name):
    if not isinstance(name, str) or not name:
        return name
    if name in NAME_MAP:
        return NAME_MAP[name]
    return name_map_norm.get(normalize_name(name), name)


name_map_table = (
    pd.DataFrame(
        [{"name_variant": k, "name_canonical": v} for k, v in NAME_MAP.items()]
    )
    .sort_values("name_variant")
    .reset_index(drop=True)
)

print("NAME_MAP entries:", len(NAME_MAP))
display(name_map_table.head())


NAME_MAP entries: 128


,name_variant,name_canonical
0,Aditya Thakur,Aditya V. Thakur
1,Alastair Donaldson,Alastair F. Donaldson
2,Alexander Lew,Alexander K. Lew
3,Amir Kafshdar Goharshady,Amir K. Goharshady
4,Andrew Pitts,Andrew M. Pitts


## 3 - Load Data

In [14]:
hotcrp = pd.read_parquet(intermediate_dir / "hotcrp_members.parquet")
pc = pd.read_parquet(intermediate_dir / "researchr_pc_members.parquet")
external = pd.read_parquet(intermediate_dir / "researchr_external_members.parquet")

print("HotCRP:", hotcrp.shape)
print("Researchr PC:", pc.shape)
print("Researchr external:", external.shape)


HotCRP: (3032, 8)
Researchr PC: (2180, 12)
Researchr external: (778, 12)


## 4 - Apply Name Map

In [15]:
def canonical_name(name):
    mapped = lookup_name(name)
    if isinstance(mapped, str):
        return mapped
    return name


def name_signature(name_norm):
    return " ".join(sorted(name_norm.split()))


sources = {
    "HotCRP": hotcrp,
    "Researchr PC": pc,
    "Researchr external": external,
}

for label, df in sources.items():
    df["name_original"] = df["name"]
    df["name_canonical"] = df["name"].map(canonical_name)
    df["name_norm"] = df["name_canonical"].map(normalize_name)
    df["changed_by_name_map"] = df["name_original"] != df["name_canonical"]
    print(label, "changed by NAME_MAP:", int(df["changed_by_name_map"].sum()))

hotcrp[["conference", "year", "name_original", "name_canonical", "name_norm"]].head()


HotCRP changed by NAME_MAP: 233
Researchr PC changed by NAME_MAP: 27
Researchr external changed by NAME_MAP: 8


,conference,year,name_original,name_canonical,name_norm
0,ICFP,2017,Adam Chlipala,Adam Chlipala,adam chlipala
1,ICFP,2017,Alan Jeffrey,Alan Jeffrey,alan jeffrey
2,ICFP,2017,Alexandra Silva,Alexandra Silva,alexandra silva
3,ICFP,2017,Ben Lippmeier,Ben Lippmeier,ben lippmeier
4,ICFP,2017,Beta Ziliani,Beta Ziliani,beta ziliani


## 5 - Create Comparison Table

In [16]:
def source_group(df, source_name):
    out = (
        df.groupby(["conference", "year", "name_norm"], as_index=False)
          .agg(
              name=("name_original", lambda x: "; ".join(sorted(set(x)))),
              name_canonical=("name_canonical", lambda x: "; ".join(sorted(set(x)))),
              affiliation=("affiliation", lambda x: "; ".join(sorted(set(str(i) for i in x if pd.notna(i))))),
          )
    )
    out[f"on_{source_name}"] = True
    out = out.rename(columns={
        "name": f"name_{source_name}",
        "name_canonical": f"name_canonical_{source_name}",
        "affiliation": f"affiliation_{source_name}",
    })
    return out


hot = source_group(hotcrp, "hotcrp")
pc_g = source_group(pc, "pc")
epc_g = source_group(external.query("committee_type == 'EPC'"), "epc")
erc_g = source_group(external.query("committee_type == 'ERC'"), "erc")

keys = pd.concat([
    hot[["conference", "year", "name_norm"]],
    pc_g[["conference", "year", "name_norm"]],
    epc_g[["conference", "year", "name_norm"]],
    erc_g[["conference", "year", "name_norm"]],
], ignore_index=True).drop_duplicates()

comparison = (
    keys.merge(hot, on=["conference", "year", "name_norm"], how="left")
        .merge(pc_g, on=["conference", "year", "name_norm"], how="left")
        .merge(epc_g, on=["conference", "year", "name_norm"], how="left")
        .merge(erc_g, on=["conference", "year", "name_norm"], how="left")
)

for col in ["on_hotcrp", "on_pc", "on_epc", "on_erc"]:
    comparison[col] = comparison[col].fillna(False).astype(bool)

name_cols = [
    "name_canonical_hotcrp",
    "name_canonical_pc",
    "name_canonical_epc",
    "name_canonical_erc",
]
comparison["name_canonical"] = comparison[name_cols].bfill(axis=1).iloc[:, 0]
comparison["on_researchr_any"] = comparison["on_pc"] | comparison["on_epc"] | comparison["on_erc"]

comparison["category"] = "other"
comparison.loc[comparison.on_hotcrp & comparison.on_researchr_any, "category"] = "HotCRP and Researchr"
comparison.loc[comparison.on_hotcrp & ~comparison.on_researchr_any, "category"] = "HotCRP only"
comparison.loc[~comparison.on_hotcrp & comparison.on_researchr_any, "category"] = "Researchr only"

comparison = comparison.sort_values(["conference", "year", "name_norm"]).reset_index(drop=True)
print(comparison.shape)
display(comparison.head())


(3029, 22)


,conference,year,name_norm,name_hotcrp,name_canonical_hotcrp,affiliation_hotcrp,on_hotcrp,name_pc,name_canonical_pc,affiliation_pc,...,name_canonical_epc,affiliation_epc,on_epc,name_erc,name_canonical_erc,affiliation_erc,on_erc,name_canonical,on_researchr_any,category
0,ICFP,2017,adam chlipala,Adam Chlipala,Adam Chlipala,MIT CSAIL,True,Adam Chlipala,Adam Chlipala,"Massachusetts Institute of Technology, USA",...,NaN,NaN,False,NaN,NaN,NaN,False,Adam Chlipala,True,HotCRP and Researchr
1,ICFP,2017,alan jeffrey,Alan Jeffrey,Alan Jeffrey,Mozilla Research,True,Alan Jeffrey,Alan Jeffrey,Mozilla Research,...,NaN,NaN,False,NaN,NaN,NaN,False,Alan Jeffrey,True,HotCRP and Researchr
2,ICFP,2017,alexandra silva,Alexandra Silva,Alexandra Silva,University College London,True,Alexandra Silva,Alexandra Silva,University College London,...,NaN,NaN,False,NaN,NaN,NaN,False,Alexandra Silva,True,HotCRP and Researchr
3,ICFP,2017,ben lippmeier,Ben Lippmeier,Ben Lippmeier,Digital Asset,True,Ben Lippmeier,Ben Lippmeier,Digital Asset / UNSW Australia,...,NaN,NaN,False,NaN,NaN,NaN,False,Ben Lippmeier,True,HotCRP and Researchr
4,ICFP,2017,beta ziliani,Beta Ziliani,Beta Ziliani,"CONICET and FAMAF, Universidad Nacional de Cór...",True,Beta Ziliani,Beta Ziliani,"FAMAF, UNC and CONICET",...,NaN,NaN,False,NaN,NaN,NaN,False,Beta Ziliani,True,HotCRP and Researchr


## 6 - Cell Level Overlap

In [17]:
def count_true(s):
    return int(s.sum())


summary_rows = []

for (conf, year), g in comparison.groupby(["conference", "year"]):
    summary_rows.append({
        "conference": conf,
        "year": int(year),
        "n_hotcrp": count_true(g["on_hotcrp"]),
        "n_pc": count_true(g["on_pc"]),
        "n_epc": count_true(g["on_epc"]),
        "n_erc": count_true(g["on_erc"]),
        "n_researchr_any": count_true(g["on_researchr_any"]),
        "n_hotcrp_in_pc": count_true(g["on_hotcrp"] & g["on_pc"]),
        "n_hotcrp_in_epc": count_true(g["on_hotcrp"] & g["on_epc"]),
        "n_hotcrp_in_erc": count_true(g["on_hotcrp"] & g["on_erc"]),
        "n_hotcrp_in_researchr_any": count_true(g["on_hotcrp"] & g["on_researchr_any"]),
        "n_hotcrp_only": count_true(g["on_hotcrp"] & ~g["on_researchr_any"]),
        "n_researchr_only": count_true(~g["on_hotcrp"] & g["on_researchr_any"]),
    })

summary = pd.DataFrame(summary_rows).sort_values(["conference", "year"])
summary["share_hotcrp_in_researchr_any"] = (
    summary["n_hotcrp_in_researchr_any"] / summary["n_hotcrp"]
).round(3)

display(summary)


,conference,year,n_hotcrp,n_pc,n_epc,n_erc,n_researchr_any,n_hotcrp_in_pc,n_hotcrp_in_epc,n_hotcrp_in_erc,n_hotcrp_in_researchr_any,n_hotcrp_only,n_researchr_only,share_hotcrp_in_researchr_any
0,ICFP,2017,23,23,0,0,23,23,0,0,23,0,0,1.000
1,ICFP,2018,63,18,0,43,61,18,0,43,61,2,0,0.968
2,ICFP,2019,72,20,0,51,71,20,0,51,71,1,0,0.986
3,ICFP,2020,60,17,0,42,59,17,0,42,59,1,0,0.983
4,ICFP,2021,30,30,0,0,30,30,0,0,30,0,0,1.000
5,ICFP,2022,42,42,0,0,42,42,0,0,42,0,0,1.000
6,ICFP,2023,55,54,0,0,54,54,0,0,54,1,0,0.982
7,ICFP,2024,51,51,0,0,51,51,0,0,51,0,0,1.000
8,ICFP,2025,63,63,0,0,63,63,0,0,63,0,0,1.000
9,OOPSLA,2017,59,31,28,0,59,31,28,0,59,0,0,1.000


## 7 - ICFP 2018 Example

In [18]:
icfp2018 = comparison.query("conference == 'ICFP' and year == 2018").copy()

print("ICFP 2018")
print("HotCRP:", int(icfp2018.on_hotcrp.sum()))
print("Researchr PC:", int(icfp2018.on_pc.sum()))
print("Researchr EPC:", int(icfp2018.on_epc.sum()))
print("Researchr ERC:", int(icfp2018.on_erc.sum()))
print("Researchr PC + EPC + ERC:", int(icfp2018.on_researchr_any.sum()))
print("HotCRP ∩ Researchr:", int((icfp2018.on_hotcrp & icfp2018.on_researchr_any).sum()))
print("HotCRP only:", int((icfp2018.on_hotcrp & ~icfp2018.on_researchr_any).sum()))
print("Researchr only:", int((~icfp2018.on_hotcrp & icfp2018.on_researchr_any).sum()))


ICFP 2018
HotCRP: 63
Researchr PC: 18
Researchr EPC: 0
Researchr ERC: 43
Researchr PC + EPC + ERC: 61
HotCRP ∩ Researchr: 61
HotCRP only: 2
Researchr only: 0


In [19]:
icfp2018_mismatch = icfp2018.query(
    "(on_hotcrp and not on_researchr_any) or ((not on_hotcrp) and on_researchr_any)"
).copy()

display(icfp2018_mismatch[
    [
        "name_norm", "name_canonical", "on_hotcrp", "on_pc", "on_epc", "on_erc",
        "name_hotcrp", "name_pc", "name_epc", "name_erc",
        "name_canonical_hotcrp", "name_canonical_pc", "name_canonical_epc", "name_canonical_erc",
    ]
])


,name_norm,name_canonical,on_hotcrp,on_pc,on_epc,on_erc,name_hotcrp,name_pc,name_epc,name_erc,name_canonical_hotcrp,name_canonical_pc,name_canonical_epc,name_canonical_erc
71,ryan r newton,Ryan R. Newton,True,False,False,False,Ryan Newton,NaN,NaN,NaN,Ryan R. Newton,NaN,NaN,NaN
75,simon marlow,Simon Marlow,True,False,False,False,Simon Marlow,NaN,NaN,NaN,Simon Marlow,NaN,NaN,NaN


## 8 - Save Tables

In [20]:
comparison.to_parquet(intermediate_dir / "hotcrp_researchr_name_comparison.parquet", index=False)
summary.to_csv(check_tables_dir / "hotcrp_researchr_overlap_summary.csv", index=False)
icfp2018_mismatch.to_csv(check_tables_dir / "icfp2018_hotcrp_researchr_mismatches.csv", index=False)
name_map_table.to_csv(dependency_tables_dir / "name_map_used_for_source_comparison.csv", index=False)

print("saved comparison and summary tables")


saved comparison and summary tables


## 9 - What I learned

For ICFP 2018, after applying `NAME_MAP`, HotCRP is closest to the union of
Researchr PC and ERC.

The remaining mismatches are the rows I need to inspect as real source
differences, not just spelling differences.
